# 06 — SQL: Beginner to Advanced

**Practical Jupyter Notebook | Database design • Queries • Joins • Advanced SQL**

This notebook uses Python's built-in `sqlite3` module, so **no database server or installation is required**. Execute cells in order. SQL syntax is mostly portable; notes flag MySQL and SQLite differences. SQL examples are executed using `pd.read_sql_query()` for readable tables.

## 0. Setup: Our sample database
We use `departments` and `employees` throughout. Each employee optionally belongs to a department; an employee can have a manager who is another employee. A department with no employees helps illustrate outer joins. **Run this setup cell first.**

In [ ]:
import sqlite3
import pandas as pd
conn = sqlite3.connect(":memory:")
conn.execute("PRAGMA foreign_keys = ON")
conn.executescript("""
CREATE TABLE departments (
    dept_id INTEGER PRIMARY KEY,
    dept_name TEXT NOT NULL UNIQUE,
    location TEXT NOT NULL DEFAULT 'Remote'
);
CREATE TABLE employees (
    emp_id INTEGER PRIMARY KEY,
    emp_name TEXT NOT NULL,
    dept_id INTEGER,
    salary REAL NOT NULL CHECK (salary >= 0),
    hire_date TEXT NOT NULL,
    manager_id INTEGER,
    FOREIGN KEY (dept_id) REFERENCES departments(dept_id),
    FOREIGN KEY (manager_id) REFERENCES employees(emp_id)
);
INSERT INTO departments VALUES
(1,'Engineering','Chennai'), (2,'Analytics','Bengaluru'),
(3,'HR','Coimbatore'), (4,'Sales','Mumbai'), (5,'Legal','Delhi');
INSERT INTO employees VALUES
(1,'Asha',1,90000,'2021-01-10',NULL),
(2,'Bala',1,70000,'2022-06-01',1),
(3,'Charu',2,85000,'2020-03-15',NULL),
(4,'Deepak',2,65000,'2023-02-20',3),
(5,'Esha',2,65000,'2023-08-01',3),
(6,'Farhan',3,50000,'2024-04-12',NULL),
(7,'Gita',4,55000,'2022-11-05',NULL),
(8,'Hari',NULL,45000,'2025-01-02',NULL);
""")
def query(sql):
    return pd.read_sql_query(sql, conn)
query("SELECT * FROM employees")

# 1. Database design

### 1.1 Database
A **database** is an organized collection of related data. A relational database stores data in linked tables. A **DBMS** (e.g., MySQL, PostgreSQL, SQLite) manages storage, queries, transactions and permissions.

### 1.2 Tables
A **table** contains rows (records) and columns (attributes). `employees` has columns such as `emp_id`, `emp_name` and `salary`; each row represents one employee.

### 1.3 Primary key
A **PRIMARY KEY** uniquely identifies each row. Its values cannot be NULL or duplicated. A table can have a composite primary key spanning multiple columns.

### 1.4 Foreign key
A **FOREIGN KEY** refers to a primary key or other unique key in another (or the same) table. It helps prevent orphan records. A nullable foreign key permits employees without departments.

### 1.5 Relationships
- **One-to-one:** one user ↔ one profile (use a UNIQUE foreign key).
- **One-to-many:** one department → many employees (our example).
- **Many-to-many:** students ↔ courses; implement with an enrollment junction table containing both foreign keys.
- **Self-referential:** employee → manager (both rows in `employees`).

### 1.6 Constraints
`NOT NULL` requires a value; `UNIQUE` prohibits duplicates; `PRIMARY KEY` uniquely identifies rows; `FOREIGN KEY` enforces references; `CHECK` enforces conditions; `DEFAULT` supplies a value when omitted. Constraint behavior can vary across DBMSs. **Normalization** reduces duplication: separate departments from employees instead of repeating department details in every employee row.

In [ ]:
# Inspect the schema and constraints
query("SELECT name, sql FROM sqlite_master WHERE type='table' AND name NOT LIKE 'sqlite_%'")

### DDL: creating a many-to-many relationship
`CREATE TABLE`, `ALTER TABLE` and `DROP TABLE` are **DDL** (Data Definition Language). `INSERT`, `UPDATE`, `DELETE` are **DML** (Data Manipulation Language). `SELECT` retrieves data.

In [ ]:
conn.executescript("""
CREATE TABLE IF NOT EXISTS courses (
    course_id INTEGER PRIMARY KEY,
    course_name TEXT NOT NULL UNIQUE
);
CREATE TABLE IF NOT EXISTS enrollments (
    emp_id INTEGER NOT NULL,
    course_id INTEGER NOT NULL,
    enrolled_on TEXT DEFAULT CURRENT_DATE,
    PRIMARY KEY (emp_id, course_id),
    FOREIGN KEY (emp_id) REFERENCES employees(emp_id),
    FOREIGN KEY (course_id) REFERENCES courses(course_id)
);
""")
query("PRAGMA table_info(enrollments)")

# 2. Basic SQL queries

### 2.1 SELECT
Choose columns with `SELECT`; `*` selects all columns. Prefer explicit columns in production to avoid unexpected schema changes.

In [ ]:
query("SELECT emp_id, emp_name, salary FROM employees")

### 2.2 WHERE
Filter rows **before aggregation**. Common operators: `=`, `<>`, `>`, `<`, `>=`, `<=`, `AND`, `OR`, `NOT`, `IN`, `BETWEEN`, `LIKE`, `IS NULL`. `NULL` is unknown: use `IS NULL`, not `= NULL`.

In [ ]:
query("""SELECT emp_name, salary, dept_id FROM employees
WHERE salary >= 65000 AND dept_id IN (1, 2)""")

In [ ]:
query("SELECT emp_name FROM employees WHERE emp_name LIKE 'A%' OR dept_id IS NULL")

### 2.3 ORDER BY
Sort ascending (`ASC`, default) or descending (`DESC`). Multiple columns break ties. Without `ORDER BY`, row order is not guaranteed.

In [ ]:
query("SELECT emp_name, dept_id, salary FROM employees ORDER BY dept_id ASC, salary DESC")

### 2.4 GROUP BY
Combine rows into groups, typically with aggregates: `COUNT`, `SUM`, `AVG`, `MIN`, `MAX`. Every nonaggregated selected column should be grouped (or functionally dependent on grouped columns, where supported).

In [ ]:
query("""SELECT dept_id, COUNT(*) AS employee_count,
ROUND(AVG(salary), 2) AS avg_salary, MAX(salary) AS highest_salary
FROM employees GROUP BY dept_id""")

### 2.5 HAVING
Filter **groups after aggregation**. `WHERE salary > 50000` filters individual rows; `HAVING AVG(salary) > 60000` filters department groups.

In [ ]:
query("""SELECT dept_id, COUNT(*) AS headcount, AVG(salary) AS avg_salary
FROM employees WHERE dept_id IS NOT NULL
GROUP BY dept_id HAVING AVG(salary) > 60000""")

### 2.6 DISTINCT
Remove duplicate combinations of the selected columns. `COUNT(DISTINCT dept_id)` counts distinct non-NULL departments.

In [ ]:
query("SELECT DISTINCT dept_id FROM employees ORDER BY dept_id")

### 2.7 LIMIT
Restrict returned rows. Combine with `ORDER BY` for a meaningful top-N result. MySQL and SQLite support `LIMIT`; SQL Server uses `TOP` or `OFFSET … FETCH`.

In [ ]:
query("SELECT emp_name, salary FROM employees ORDER BY salary DESC, emp_id LIMIT 3")

### SQL logical processing order
Conceptually: **FROM / JOIN → WHERE → GROUP BY → HAVING → SELECT → DISTINCT → ORDER BY → LIMIT**. This explains why an aggregate is normally filtered using `HAVING`, not `WHERE`. The database optimizer may execute operations differently.

# 3. Joins
A **JOIN** combines rows from multiple tables using a related key. Aliases (`e`, `d`) make queries easier to read.

### 3.1 INNER JOIN
Return only employees with a matching department.

In [ ]:
query("""SELECT e.emp_name, d.dept_name
FROM employees AS e
INNER JOIN departments AS d ON e.dept_id = d.dept_id""")

### 3.2 LEFT JOIN
Return **all left-side employees**, including Hari whose department is NULL. Missing right-side values become NULL.

In [ ]:
query("""SELECT e.emp_name, d.dept_name
FROM employees e LEFT JOIN departments d ON e.dept_id = d.dept_id
ORDER BY e.emp_id""")

### 3.3 RIGHT JOIN
Return **all right-side departments**, including Legal with no employees. SQLite 3.39+ supports `RIGHT JOIN`, but for compatibility reverse table order and use `LEFT JOIN`. MySQL supports `RIGHT JOIN`.

In [ ]:
# Equivalent to: employees e RIGHT JOIN departments d ON e.dept_id = d.dept_id
query("""SELECT d.dept_name, e.emp_name
FROM departments d LEFT JOIN employees e ON d.dept_id = e.dept_id
ORDER BY d.dept_id, e.emp_id""")

### 3.4 FULL OUTER JOIN
Return all matches plus unmatched rows from **both** tables (Hari and Legal). PostgreSQL and modern SQLite support `FULL OUTER JOIN`; **MySQL does not**, so emulate it using a `LEFT JOIN` plus a reversed `LEFT JOIN` for unmatched right rows.

In [ ]:
# Portable FULL OUTER JOIN emulation (also works in MySQL)
query("""SELECT e.emp_name, d.dept_name
FROM employees e LEFT JOIN departments d ON e.dept_id = d.dept_id
UNION ALL
SELECT e.emp_name, d.dept_name
FROM departments d LEFT JOIN employees e ON e.dept_id = d.dept_id
WHERE e.emp_id IS NULL""")

### 3.5 CROSS JOIN
Every left row pairs with every right row: **N × M** rows. Useful for creating combinations; potentially expensive on large tables.

In [ ]:
query("""SELECT e.emp_name, d.dept_name
FROM (SELECT emp_name FROM employees ORDER BY emp_id LIMIT 2) e
CROSS JOIN (SELECT dept_name FROM departments ORDER BY dept_id LIMIT 3) d""")

### Bonus: SELF JOIN
Join a table to itself to show employee–manager relationships. `LEFT JOIN` keeps employees with no manager.

In [ ]:
query("""SELECT e.emp_name AS employee, m.emp_name AS manager
FROM employees e LEFT JOIN employees m ON e.manager_id = m.emp_id
ORDER BY e.emp_id""")

# 4. Advanced SQL

### 4.1 UNION and UNION ALL
`UNION` combines compatible result sets and removes duplicate rows; `UNION ALL` preserves duplicates and usually avoids the deduplication cost. Both queries must return the same number of columns with compatible types. A final `ORDER BY` applies to the combined result.

In [ ]:
query("""SELECT emp_name AS person FROM employees WHERE dept_id = 1
UNION
SELECT emp_name FROM employees WHERE salary >= 85000
ORDER BY person""")

In [ ]:
query("""SELECT emp_name AS person FROM employees WHERE dept_id = 1
UNION ALL
SELECT emp_name FROM employees WHERE salary >= 85000""")

### 4.2 CASE
Conditional logic inside SQL: `CASE WHEN condition THEN value … ELSE value END`. If no condition matches and `ELSE` is omitted, the result is NULL.

In [ ]:
query("""SELECT emp_name, salary,
CASE WHEN salary >= 80000 THEN 'High'
     WHEN salary >= 60000 THEN 'Medium'
     ELSE 'Entry' END AS salary_band
FROM employees ORDER BY salary DESC""")

### 4.3 Subqueries
A **subquery** is a query inside another query. Common forms: scalar subquery (one value), `IN` (membership), correlated subquery (references outer row), and `EXISTS` (tests whether at least one matching row exists).

In [ ]:
# Scalar subquery: employees earning above the company average
query("""SELECT emp_name, salary FROM employees
WHERE salary > (SELECT AVG(salary) FROM employees)""")

In [ ]:
# IN subquery: employees in departments located in Chennai or Bengaluru
query("""SELECT emp_name FROM employees
WHERE dept_id IN (SELECT dept_id FROM departments
                  WHERE location IN ('Chennai','Bengaluru'))""")

In [ ]:
# Correlated subquery: earn more than own department average
query("""SELECT e.emp_name, e.dept_id, e.salary FROM employees e
WHERE e.salary > (SELECT AVG(e2.salary) FROM employees e2
                  WHERE e2.dept_id = e.dept_id)""")

In [ ]:
# EXISTS: departments with at least one employee
query("""SELECT d.dept_name FROM departments d
WHERE EXISTS (SELECT 1 FROM employees e WHERE e.dept_id = d.dept_id)""")

### 4.4 Common Table Expressions (CTEs)
`WITH name AS (SELECT …)` names an intermediate result for the **following single SQL statement**. It improves readability and enables reuse within that statement. CTEs are not automatically materialized; optimization depends on the database. Recursive CTEs can traverse hierarchies.

In [ ]:
query("""WITH dept_stats AS (
    SELECT dept_id, AVG(salary) AS avg_salary, COUNT(*) AS headcount
    FROM employees WHERE dept_id IS NOT NULL GROUP BY dept_id
)
SELECT d.dept_name, s.headcount, ROUND(s.avg_salary, 2) AS avg_salary
FROM dept_stats s JOIN departments d ON s.dept_id = d.dept_id
WHERE s.headcount >= 2""")

In [ ]:
# Recursive CTE: generate the integers 1–5
query("""WITH RECURSIVE numbers(n) AS (
    SELECT 1
    UNION ALL
    SELECT n + 1 FROM numbers WHERE n < 5
)
SELECT n FROM numbers""")

### 4.5 Views
A **view** stores a query definition and behaves like a virtual table. Ordinary views do **not** store query results. They simplify reuse and can help expose selected columns. Some DBMSs offer materialized views, which store results and need refreshes.

In [ ]:
conn.execute("""CREATE VIEW IF NOT EXISTS employee_directory AS
SELECT e.emp_id, e.emp_name, d.dept_name, e.salary
FROM employees e LEFT JOIN departments d ON e.dept_id = d.dept_id""")
query("SELECT * FROM employee_directory ORDER BY emp_id")

### 4.6 Indexes
An **index** is a search structure that can accelerate filtering, joins and sorting. It consumes storage and adds overhead to inserts/updates/deletes. Index frequently filtered or joined columns; inspect query plans rather than indexing everything. Primary keys and UNIQUE constraints often create indexes automatically.

In [ ]:
conn.execute("CREATE INDEX IF NOT EXISTS idx_employees_dept_salary ON employees(dept_id, salary)")
query("EXPLAIN QUERY PLAN SELECT * FROM employees WHERE dept_id = 2 AND salary > 60000")

### 4.7 Window functions
A window function computes across related rows **without collapsing them into one row per group**. `OVER (PARTITION BY ... ORDER BY ...)` defines the window. `PARTITION BY` restarts calculations for each group; `ORDER BY` defines the sequence. Include a unique tie-breaker for deterministic row numbering.

**ROW_NUMBER()** assigns unique sequential positions even when values tie.

In [ ]:
query("""SELECT emp_name, dept_id, salary,
ROW_NUMBER() OVER (PARTITION BY dept_id ORDER BY salary DESC, emp_id) AS row_num
FROM employees WHERE dept_id IS NOT NULL""")

**RANK()** gives ties the same rank and leaves gaps (1, 2, 2, 4). **DENSE_RANK()** gives ties the same rank without gaps (1, 2, 2, 3). The ranking `ORDER BY` below uses only salary so tied salaries remain tied.

In [ ]:
query("""SELECT emp_name, dept_id, salary,
RANK() OVER (PARTITION BY dept_id ORDER BY salary DESC) AS salary_rank,
DENSE_RANK() OVER (PARTITION BY dept_id ORDER BY salary DESC) AS dense_salary_rank
FROM employees WHERE dept_id IS NOT NULL
ORDER BY dept_id, salary DESC, emp_id""")

**LAG()** accesses an earlier row in the window; **LEAD()** accesses a later row. Useful for comparing consecutive purchases, time-series values or employee hire dates. Default offset is 1; rows without a predecessor/successor return NULL unless a default is provided.

In [ ]:
query("""SELECT emp_name, dept_id, hire_date, salary,
LAG(salary) OVER (PARTITION BY dept_id ORDER BY hire_date, emp_id) AS previous_salary,
LEAD(salary) OVER (PARTITION BY dept_id ORDER BY hire_date, emp_id) AS next_salary
FROM employees WHERE dept_id IS NOT NULL
ORDER BY dept_id, hire_date, emp_id""")

### Practical pattern: top 2 earners per department
A window function is not generally allowed directly in `WHERE`, so calculate rankings in a CTE and filter in the outer query. Use `ROW_NUMBER` for **exactly up to two rows** per department; `RANK` can return more than two when salaries tie.

In [ ]:
query("""WITH ranked AS (
    SELECT emp_name, dept_id, salary,
           ROW_NUMBER() OVER (PARTITION BY dept_id ORDER BY salary DESC, emp_id) AS rn
    FROM employees WHERE dept_id IS NOT NULL
)
SELECT * FROM ranked WHERE rn <= 2 ORDER BY dept_id, rn""")

# 5. Practical SQL patterns for analytics

### Conditional aggregation
Count how many employees in each department earn at least 65,000. `CASE` works in SQLite, MySQL and PostgreSQL.

In [ ]:
query("""SELECT d.dept_name, COUNT(e.emp_id) AS total_employees,
SUM(CASE WHEN e.salary >= 65000 THEN 1 ELSE 0 END) AS employees_65k_plus
FROM departments d LEFT JOIN employees e ON d.dept_id = e.dept_id
GROUP BY d.dept_id, d.dept_name ORDER BY d.dept_id""")

### NULL handling
`COALESCE(a, b)` returns the first non-NULL argument. In outer joins, `COUNT(*)` counts the unmatched placeholder row, whereas `COUNT(e.emp_id)` correctly returns zero employees for Legal.

In [ ]:
query("""SELECT d.dept_name, COUNT(e.emp_id) AS headcount,
COALESCE(ROUND(AVG(e.salary), 2), 0) AS average_salary
FROM departments d LEFT JOIN employees e ON d.dept_id = e.dept_id
GROUP BY d.dept_id, d.dept_name ORDER BY d.dept_id""")

# 6. MySQL vs SQLite quick reference

| Feature | SQLite (this notebook) | MySQL 8+ |
|---|---|---|
| Connect | `sqlite3.connect(':memory:')` | MySQL server + driver |
| Integer auto-generated PK | `INTEGER PRIMARY KEY` | `INT AUTO_INCREMENT PRIMARY KEY` |
| FULL OUTER JOIN | SQLite 3.39+ | Emulate using LEFT JOIN + UNION ALL |
| RIGHT JOIN | SQLite 3.39+ | Supported |
| LIMIT | Supported | Supported |
| CTE and windows | Modern SQLite | MySQL 8+ |
| Date arithmetic | `date(...)`, `strftime(...)` | `DATE_ADD`, `DATE_FORMAT` |
| Explain plan | `EXPLAIN QUERY PLAN ...` | `EXPLAIN ...` |

**Important:** SQLite is permissive about some `GROUP BY` queries. Write portable SQL by grouping all nonaggregated selected columns. In MySQL, `ONLY_FULL_GROUP_BY` enforces stricter grouping rules. SQLite is convenient for practice; use MySQL/PostgreSQL for DBMS-specific production features.

# 7. Practice exercises
Try these **before** expanding the answers below:

1. List employees earning between 55,000 and 75,000, highest salary first.
2. Show each department's employee count, including empty departments.
3. Find employees whose salary exceeds their own department's average.
4. Find the department with the highest average salary among nonempty departments.
5. Show the second-highest **distinct** salary in each department.
6. Show each employee and their manager's name, including employees with no manager.
7. Find departments without employees using `NOT EXISTS`.
8. For each department, show employee salaries and the previous hire's salary using `LAG`.

### Answer key

**Exercise 1 — solution**

In [ ]:
query("""SELECT emp_name, salary FROM employees WHERE salary BETWEEN 55000 AND 75000 ORDER BY salary DESC, emp_id""")

**Exercise 2 — solution**

In [ ]:
query("""SELECT d.dept_name, COUNT(e.emp_id) AS headcount FROM departments d LEFT JOIN employees e ON d.dept_id=e.dept_id GROUP BY d.dept_id, d.dept_name ORDER BY d.dept_id""")

**Exercise 3 — solution**

In [ ]:
query("""SELECT e.emp_name, e.salary FROM employees e WHERE e.salary > (SELECT AVG(x.salary) FROM employees x WHERE x.dept_id=e.dept_id)""")

**Exercise 4 — solution**

In [ ]:
query("""SELECT d.dept_name, AVG(e.salary) AS avg_salary FROM departments d JOIN employees e ON d.dept_id=e.dept_id GROUP BY d.dept_id, d.dept_name ORDER BY avg_salary DESC LIMIT 1""")

**Exercise 5 — solution**

In [ ]:
query("""WITH r AS (SELECT emp_name, dept_id, salary, DENSE_RANK() OVER (PARTITION BY dept_id ORDER BY salary DESC) AS dr FROM employees WHERE dept_id IS NOT NULL) SELECT * FROM r WHERE dr=2""")

**Exercise 6 — solution**

In [ ]:
query("""SELECT e.emp_name, m.emp_name AS manager FROM employees e LEFT JOIN employees m ON e.manager_id=m.emp_id""")

**Exercise 7 — solution**

In [ ]:
query("""SELECT d.dept_name FROM departments d WHERE NOT EXISTS (SELECT 1 FROM employees e WHERE e.dept_id=d.dept_id)""")

**Exercise 8 — solution**

In [ ]:
query("""SELECT emp_name, dept_id, hire_date, salary, LAG(salary) OVER (PARTITION BY dept_id ORDER BY hire_date, emp_id) AS previous_hire_salary FROM employees WHERE dept_id IS NOT NULL""")

# 8. Summary / interview revision

- **Primary key vs foreign key:** unique row identifier vs reference to a key in another/self table.
- **WHERE vs HAVING:** row filter before grouping vs group filter after aggregation.
- **INNER vs LEFT:** matched rows only vs all left rows plus matches.
- **UNION vs UNION ALL:** remove duplicates vs keep duplicates.
- **CTE vs view:** named result for one statement vs saved query definition.
- **GROUP BY vs window function:** collapses rows vs preserves row-level detail.
- **ROW_NUMBER vs RANK vs DENSE_RANK:** unique sequence vs tied ranks with gaps vs tied ranks without gaps.
- **LAG vs LEAD:** previous row vs next row within the ordered window.
- **Indexes:** faster reads in appropriate queries, but cost storage and write time.

**Practice tip:** change sample rows and rerun queries; add duplicate salaries, NULL departments, and empty departments to test edge cases.